# Run NMME Nino3.4 diagnostics

This notebook generates the NMME benchmark diagnostics used by `4_refactor_sst_skill_ts.ipynb`.

It calls `scripts/run_nmme_nino34_yeager_diag.py`, which reads the member-split NMME SST archive, computes Nino3.4 anomalies following the Yeager f03/f04-style workflow, evaluates ACC and nRMSE against HadISST2, and writes the benchmark NetCDF consumed by the skill-score overlay.

Primary output:

`/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/NMME_Nino34_skill_<DATA_START>_<DATA_END>.nc`

In [15]:
import os
import re
import subprocess
import sys
from pathlib import Path

import xarray as xr

# Identify repository root. This works when the notebook is run from either
# the repository root or the jupyter/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPT_PATH = REPO_ROOT / "scripts" / "run_nmme_nino34_yeager_diag.py"
print(f"Repository root: {REPO_ROOT}")
print(f"Python         : {sys.executable}")
print(f"Script         : {SCRIPT_PATH}")

Repository root: /global/u2/z/zhan391/code/ESP-Lab
Python         : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
Script         : /global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py


## Configuration

Set `MODELS` to an explicit list of NMME model directory names when you want to process only selected models. Leave `MODELS = []` to use `MODEL_SET = "all"` for every downloaded SST model or `MODEL_SET = "yeager-f03"` for the smaller eight-model Yeager-style subset. Set `FORCE = True` only when you want to rebuild cached per-model Nino3.4 anomaly files.

In [16]:
NMME_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member")
OBS_FILE = Path(
    "/global/cfs/cdirs/e3sm/diagnostics/observations/Atm/time-series/"
    "HadISST2/sst_186901_202212.nc"
)
OUTDIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME")
FIGDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")

OBS_VAR = "sst"
MODEL_SET = "all"  # Used only when MODELS is empty: "all" or "yeager-f03".

# Explicit model names to process. Use [] to fall back to MODEL_SET.
MODELS = [
    "CanSIPS-IC3",
    "GFDL-CM2p1",
    "NASA-GMAO-062012",
    "CanSIPS-IC4",
    "GFDL-CM2p1-aer04",
    "NCAR-CESM1",
    "CanSIPSv2",
    "GFDL-CM2p5-FLOR-A06",
    "NCEP-CFSv1",
    "CMC1-CanCM3",
    "GFDL-CM2p5-FLOR-B01",
    "NCEP-CFSv2",
    "CMC2-CanCM4",
    "GFDL-SPEAR",
    "GEM-NEMO",
    "NASA-GMAO",
    "CanCM4i",
    "NASA-GEOSS2S",
    "COLA-RSMAS-CCSM3",
    "IRI-ECHAM4p5-AnomalyCoupled",
    "COLA-RSMAS-CCSM4",
    "IRI-ECHAM4p5-DirectCoupled",
    "COLA-RSMAS-CESM1",
]
DATA_START = 1980  # Use "auto" for earliest selected-model year, or set an integer year.
DATA_END = 2020    # Use "auto" for latest selected-model year, or set an integer year.
CLIM_START = 1981
CLIM_END = 2010
FORCE = True

YEAGER_F03_MODELS = [
    "CMC1-CanCM3",
    "CMC2-CanCM4",
    "COLA-RSMAS-CCSM4",
    "GFDL-CM2p1-aer04",
    "GFDL-CM2p5-FLOR-A06",
    "GFDL-CM2p5-FLOR-B01",
    "NASA-GMAO-062012",
    "NCEP-CFSv2",
]

## Validate inputs

In [17]:
missing = []
for label, path in {
    "script": SCRIPT_PATH,
    "NMME root": NMME_ROOT,
    "HadISST2 obs file": OBS_FILE,
}.items():
    if not path.exists():
        missing.append(f"{label}: {path}")

if MODEL_SET not in {"all", "yeager-f03"}:
    missing.append(f"MODEL_SET must be 'all' or 'yeager-f03', got {MODEL_SET!r}")

if missing:
    raise FileNotFoundError("Missing or invalid configuration:\n" + "\n".join(missing))

available_models = sorted(
    p.name for p in NMME_ROOT.iterdir()
    if p.is_dir() and p.name != "logs" and (p / "sst").is_dir()
)

EXPLICIT_MODELS = list(
    dict.fromkeys(str(model).strip() for model in MODELS if str(model).strip())
)
selection_errors = []
if EXPLICIT_MODELS:
    unavailable_models = sorted(set(EXPLICIT_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODELS contains names that are not available under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = EXPLICIT_MODELS
    model_selection = "explicit MODELS list"
elif MODEL_SET == "all":
    SELECTED_MODELS = available_models
    model_selection = "MODEL_SET='all'"
else:
    unavailable_models = sorted(set(YEAGER_F03_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODEL_SET='yeager-f03' includes unavailable models under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = YEAGER_F03_MODELS
    model_selection = "MODEL_SET='yeager-f03'"

if selection_errors:
    raise ValueError("Invalid model selection:\n" + "\n".join(selection_errors))

s_chunk_pattern = re.compile(r"_S(\d+)-(\d+)\.nc$")

def _chunk_range(path):
    match = s_chunk_pattern.search(path.name)
    return (int(match.group(1)), int(match.group(2))) if match else None

def _decode_s_edge(path, first=True):
    with xr.open_dataset(path, decode_times=False) as ds:
        s_ds = ds[["S"]].copy()
    if s_ds["S"].attrs.get("calendar") == "360":
        s_ds["S"].attrs["calendar"] = "360_day"
    decoded = xr.decode_cf(s_ds, decode_times=True)["S"]
    return decoded.values[0 if first else -1]

def _model_s_years(model):
    files = []
    for path in (NMME_ROOT / model / "sst").glob("M*/*.nc"):
        chunk_range = _chunk_range(path)
        if chunk_range is not None:
            files.append((*chunk_range, path))
    if not files:
        raise FileNotFoundError(f"No SST chunks found for {model}")
    first_file = min(files, key=lambda item: (item[0], item[1]))[2]
    last_file = max(files, key=lambda item: (item[1], item[0]))[2]
    return _decode_s_edge(first_file, first=True).year, _decode_s_edge(last_file, first=False).year

model_year_ranges = {model: _model_s_years(model) for model in SELECTED_MODELS}
DATA_START_YEAR = min(start for start, _ in model_year_ranges.values()) if DATA_START == "auto" else int(DATA_START)
DATA_END_YEAR = max(end for _, end in model_year_ranges.values()) if DATA_END == "auto" else int(DATA_END)
if DATA_START_YEAR > DATA_END_YEAR:
    raise ValueError(f"DATA_START ({DATA_START_YEAR}) must be <= DATA_END ({DATA_END_YEAR})")

EXPECTED_SKILL_FILE = OUTDIR / f"NMME_Nino34_skill_{DATA_START_YEAR}_{DATA_END_YEAR}.nc"

print(f"Downloaded NMME SST model directories: {len(available_models)}")
print(available_models)
print(f"\nSelected models from {model_selection}: {len(SELECTED_MODELS)}")
print(SELECTED_MODELS)
print(f"\nSelected-model raw data window: {DATA_START_YEAR}-{DATA_END_YEAR}")
print(f"Skill climatology window: {CLIM_START}-{CLIM_END}")
print(f"Expected skill file: {EXPECTED_SKILL_FILE}")


Downloaded NMME SST model directories: 23
['CMC1-CanCM3', 'CMC2-CanCM4', 'COLA-RSMAS-CCSM3', 'COLA-RSMAS-CCSM4', 'COLA-RSMAS-CESM1', 'CanCM4i', 'CanSIPS-IC3', 'CanSIPS-IC4', 'CanSIPSv2', 'GEM-NEMO', 'GFDL-CM2p1', 'GFDL-CM2p1-aer04', 'GFDL-CM2p5-FLOR-A06', 'GFDL-CM2p5-FLOR-B01', 'GFDL-SPEAR', 'IRI-ECHAM4p5-AnomalyCoupled', 'IRI-ECHAM4p5-DirectCoupled', 'NASA-GEOSS2S', 'NASA-GMAO', 'NASA-GMAO-062012', 'NCAR-CESM1', 'NCEP-CFSv1', 'NCEP-CFSv2']

Selected models from explicit MODELS list: 23
['CanSIPS-IC3', 'GFDL-CM2p1', 'NASA-GMAO-062012', 'CanSIPS-IC4', 'GFDL-CM2p1-aer04', 'NCAR-CESM1', 'CanSIPSv2', 'GFDL-CM2p5-FLOR-A06', 'NCEP-CFSv1', 'CMC1-CanCM3', 'GFDL-CM2p5-FLOR-B01', 'NCEP-CFSv2', 'CMC2-CanCM4', 'GFDL-SPEAR', 'GEM-NEMO', 'NASA-GMAO', 'CanCM4i', 'NASA-GEOSS2S', 'COLA-RSMAS-CCSM3', 'IRI-ECHAM4p5-AnomalyCoupled', 'COLA-RSMAS-CCSM4', 'IRI-ECHAM4p5-DirectCoupled', 'COLA-RSMAS-CESM1']

Selected-model raw data window: 1980-2020
Skill climatology window: 1981-2010
Expected skill file: /glob

## Run preprocessing

This can take a while on the first run because it reads each member file and writes per-model cache files under `OUTDIR/processed/`. Later runs reuse those cached files unless `FORCE = True`.

In [18]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--nmme-root", str(NMME_ROOT),
    "--obs-file", str(OBS_FILE),
    "--obs-var", OBS_VAR,
    "--outdir", str(OUTDIR),
    "--figdir", str(FIGDIR),
    "--data-start", str(DATA_START),
    "--data-end", str(DATA_END),
    "--clim-start", str(CLIM_START),
    "--clim-end", str(CLIM_END),
]

# Pass the resolved list explicitly so the command records the exact model selection.
cmd.extend(["--models", *SELECTED_MODELS])

if FORCE:
    cmd.append("--force")

env = os.environ.copy()
conda_prefix = Path(sys.prefix)
for env_name, relpath in {
    "GDAL_DATA": "share/gdal",
    "PROJ_LIB": "share/proj",
    "PROJ_DATA": "share/proj",
}.items():
    candidate = conda_prefix / relpath
    if candidate.exists():
        env[env_name] = str(candidate)

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)

Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python /global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py --nmme-root /global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member --obs-file /global/cfs/cdirs/e3sm/diagnostics/observations/Atm/time-series/HadISST2/sst_186901_202212.nc --obs-var sst --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME --figdir /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag --data-start 1980 --data-end 2020 --clim-start 1981 --clim-end 2010 --models CanSIPS-IC3 GFDL-CM2p1 NASA-GMAO-062012 CanSIPS-IC4 GFDL-CM2p1-aer04 NCAR-CESM1 CanSIPSv2 GFDL-CM2p5-FLOR-A06 NCEP-CFSv1 CMC1-CanCM3 GFDL-CM2p5-FLOR-B01 NCEP-CFSv2 CMC2-CanCM4 GFDL-SPEAR GEM-NEMO NASA-GMAO CanCM4i NASA-GEOSS2S COLA-RSMAS-CCSM3 IRI-ECHAM4p5-AnomalyCoupled COLA-RSMAS-CCSM4 IRI-ECHAM4p5-DirectCoupled COLA-RSMAS-CESM1 --force
[NMME] data window: 1980-2020
[NMME] skill climatology window: 1981-2010
[OBS] loading HadISST2 Nino3.4
[NMME] processing 23 model(s)

CompletedProcess(args=['/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python', '/global/u2/z/zhan391/code/ESP-Lab/scripts/run_nmme_nino34_yeager_diag.py', '--nmme-root', '/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member', '--obs-file', '/global/cfs/cdirs/e3sm/diagnostics/observations/Atm/time-series/HadISST2/sst_186901_202212.nc', '--obs-var', 'sst', '--outdir', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME', '--figdir', '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag', '--data-start', '1980', '--data-end', '2020', '--clim-start', '1981', '--clim-end', '2010', '--models', 'CanSIPS-IC3', 'GFDL-CM2p1', 'NASA-GMAO-062012', 'CanSIPS-IC4', 'GFDL-CM2p1-aer04', 'NCAR-CESM1', 'CanSIPSv2', 'GFDL-CM2p5-FLOR-A06', 'NCEP-CFSv1', 'CMC1-CanCM3', 'GFDL-CM2p5-FLOR-B01', 'NCEP-CFSv2', 'CMC2-CanCM4', 'GFDL-SPEAR', 'GEM-NEMO', 'NASA-GMAO', 'CanCM4i', 'NASA-GEOSS2S', 'COLA-RSMAS-CCSM3', 'IRI-ECHAM4p5-AnomalyCoupled', 'COLA-RSMAS-CCSM4', 'IRI-ECHAM4p5-DirectCoupled', 'COLA-RSMAS-CESM1', '--

## Inspect generated skill file

In [19]:
if not EXPECTED_SKILL_FILE.is_file():
    raise FileNotFoundError(f"Expected skill file was not written: {EXPECTED_SKILL_FILE}")

skill_ds = xr.open_dataset(EXPECTED_SKILL_FILE)
print(EXPECTED_SKILL_FILE)
print(skill_ds)
print("models:")
print(skill_ds.model.values.tolist())

/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/NMME_Nino34_skill_1980_2020.nc
<xarray.Dataset> Size: 102kB
Dimensions:                      (startmonth: 4, seasonal_L: 4, model: 23,
                                  monthly_L: 12)
Coordinates:
  * startmonth                   (startmonth) int64 32B 2 5 8 11
  * seasonal_L                   (seasonal_L) int64 32B 3 6 9 12
  * model                        (model) <U27 2kB 'CanSIPS-IC3' ... 'COLA-RSM...
  * monthly_L                    (monthly_L) int64 96B 1 2 3 4 5 ... 9 10 11 12
Data variables: (12/36)
    nmme_seas_skill_corr         (startmonth, seasonal_L, model) float64 3kB ...
    nmme_seas_skill_pval         (startmonth, seasonal_L, model) float64 3kB ...
    nmme_seas_skill_rmse         (startmonth, seasonal_L, model) float64 3kB ...
    nmme_seas_skill_msss         (startmonth, seasonal_L, model) float64 3kB ...
    nmme_seas_skill_rpc          (startmonth, seasonal_L, model) float64 3kB ...
    nmme_seas_skill_sig_obs      (startm